# Graph-based multi-hop retrieval — tests

Same launcher shape as `colab_run_pipeline.ipynb`: clone the repo, install, run.

There is one test: `tests/test_pipeline_smoke.py` — does the pipeline run end to end? 40
documents, 8 queries, scaled-down models, under a minute. It says nothing about graph quality.
`tests/README.md` has the details.

**Before running:**
* Colab — *Runtime → Change runtime type → T4 GPU*. CPU works too, just slower.
* Kaggle — *Settings → Accelerator → GPU*, *Internet → On*.

Section 3 (pytest) is the quick pass/fail. Section 4 runs the same code with full logs and
writes artifacts you can open. For real graphs and real numbers, use
`colab_run_pipeline.ipynb` (one run) or `colab_run_experiments.ipynb` (a sweep).

## 1. Clone the repo

In [ ]:
REF = "main"  # branch for iteration, or a commit SHA to pin a run exactly
REPO_URL = "https://github.com/hadasy-tau/graphs_project.git"

import os

# Kaggle keeps writable state in /kaggle/working, Colab in /content, anywhere else: here.
BASE = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.getcwd())
REPO = os.path.join(BASE, "graphs_project")

if os.path.isdir(REPO):
    !cd {REPO} && git fetch --all --quiet && git checkout {REF} && git pull --ff-only || true
else:
    !git clone {REPO_URL} {REPO} && cd {REPO} && git checkout {REF}

os.chdir(REPO)  # every later cell, shell command included, runs from the repo root
!git log --oneline -1

## 2. Install dependencies

Both spaCy models: `en_core_web_sm` for `config/test_small.yaml` (the smoke test) and
`en_core_web_lg` for `config/base.yaml` (the graph construction test's default).

In [ ]:
# requirements.txt pins the en_core_web_lg 3.7.1 wheel. That pin drags spaCy back to 3.7.x
# and numpy below 2.0 with it, a downgrade the already-running kernel only picks up after a
# restart. So install everything else from the file and let spaCy fetch the model builds that
# match whatever version it resolved - same NER models, no downgrade, no restart.
lines = [l for l in open("requirements.txt").read().splitlines() if "en-core-web" not in l]
with open("/tmp/requirements-colab.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

!pip install -q -r /tmp/requirements-colab.txt
!pip install -q pytest
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

In [ ]:
# Sanity check: fail here rather than halfway through a test.
import spacy
import torch

print("torch     :", torch.__version__)
print("GPU       :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - the tests still run on CPU, just slower")

for model in ("en_core_web_sm", "en_core_web_lg"):
    spacy.load(model)
    print(f"spaCy NER : {model} OK")

try:
    import pcst_fast  # noqa: F401
    print("pcst_fast : OK (the real solver)")
except ImportError:
    print("pcst_fast : MISSING - tests/pcst_fallback.py substitutes a greedy heuristic, so the"
          " smoke test still runs. Fine for watching the flow, not for reporting numbers.")

## 3. pytest — pass/fail

The structural check: the smoke test end to end on the committed 40-document fixture. Under a
minute, and no network needed — the fixture is in the repo.

In [ ]:
!python -m pytest tests -v

## 4. Smoke test with the full log

Same test as above, run directly so its section banners print: `=== 2. preprocess ===` shows
what spaCy extracted, `=== 6. retrieval ===` compares gold against retrieved. Everything it
writes goes to `tests/results/smoke_run/`.

In [ ]:
!python tests/test_pipeline_smoke.py

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

display(pd.read_csv("tests/results/smoke_run/metrics/summary_table.csv"))

## 5. Download the artifacts

`tests/results/smoke_run/` — the smoke test's graphs, retrieval output and metrics, minus the
cached `.pt` tensors.

In [ ]:
import shutil

BUNDLE = os.path.join(BASE, "graphs_project_test_output")
shutil.rmtree(BUNDLE, ignore_errors=True)

shutil.copytree("tests/results/smoke_run", os.path.join(BUNDLE, "smoke_run"),
                ignore=shutil.ignore_patterns("*.pt"))

zip_path = shutil.make_archive(BUNDLE, "zip", BUNDLE)
print(f"{zip_path}  ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Kaggle: download it from the Output tab.")